01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
    .appName("IcebergWithSpark") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.6.1,org.postgresql:postgresql:42.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [

# Exemplo básico

In [2]:
# Exclui a tabela se existir
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas")

# Cria a tabela Vendas no catalogo, usando Iceberg
spark.sql("""
    CREATE TABLE hadoop_catalog.default.vendas (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")

DataFrame[]

In [3]:
# Incluindo dados na tabela vendas
data = [
    (1, "Produto A", 10, 15.5, "2024-11-01"),
    (2, "Produto B", 5, 22.0, "2024-11-02"),
    (3, "Produto C", 8, 30.0, "2024-11-03")
]
columns = ["id", "produto", "quantidade", "preco", "data_venda"]
df = spark.createDataFrame(data, columns)
df = df.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

# Gravamos os dados na tabela vendas
df.writeTo("hadoop_catalog.default.vendas").append()

In [4]:
# Verificamos os dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2024-11-01|
|  2|Produto B|         5| 22.0|2024-11-02|
|  3|Produto C|         8| 30.0|2024-11-03|
+---+---------+----------+-----+----------+



In [5]:
# Nova coluna desconto
spark.sql("""
    ALTER TABLE hadoop_catalog.default.vendas
    ADD COLUMN desconto DOUBLE
""")

DataFrame[]

In [6]:
# Schema Atualizado
spark.sql("DESCRIBE hadoop_catalog.default.vendas").show()

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|        id|      int|   NULL|
|   produto|   string|   NULL|
|quantidade|      int|   NULL|
|     preco|   double|   NULL|
|data_venda|     date|   NULL|
|  desconto|   double|   NULL|
+----------+---------+-------+



In [7]:
# Inserindo dados com a nova coluna
data_new = [
    (4, "Produto D", 12, 25.0, "2024-11-04", 2.5),
    (5, "Produto E", 7, 18.0, "2024-11-05", 1.0)
]
columns_new = ["id", "produto", "quantidade", "preco", "data_venda", "desconto"]
df_new = spark.createDataFrame(data_new, columns_new)
df_new = df_new.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

# Gravando os dados
df_new.writeTo("hadoop_catalog.default.vendas").append()

In [8]:
# Consultando
spark.sql("SELECT * FROM hadoop_catalog.default.vendas").show()

+---+---------+----------+-----+----------+--------+
| id|  produto|quantidade|preco|data_venda|desconto|
+---+---------+----------+-----+----------+--------+
|  4|Produto D|        12| 25.0|2024-11-04|     2.5|
|  5|Produto E|         7| 18.0|2024-11-05|     1.0|
|  1|Produto A|        10| 15.5|2024-11-01|    NULL|
|  2|Produto B|         5| 22.0|2024-11-02|    NULL|
|  3|Produto C|         8| 30.0|2024-11-03|    NULL|
+---+---------+----------+-----+----------+--------+

